# Probe : bug `#r nuget:` dans dotnet-interactive sur po-2027 (#17361)

**Lane** : `myia-po-2027:CoursIA-2`, c.760, 2026-09-22
**Issue** : #17361
**Statut** : bug **réel mais non déterministe** — reproduit 3x le 2026-09-22 (c.760), non reproduit sur 2 re-tentatives le 2026-09-23 (c.803 cache chaud, c.807 cache froid — voir la cellule probe E et la RFC section Mesures). Workaround **partiel** identifié (`file.dll` marche), fix root cause **out-of-scope**.

Voir [`docs/reference/dotnet-restore-rfc-17361.md`](../../../docs/reference/dotnet-restore-rfc-17361.md) pour le diagnostic complet et l'historique des mesures.


## Portabilite (mesures first-hand c.760 + c.790 + c.807)

Le probe D utilise depuis c.803 le chemin **relatif** `#r "./_deps/QuikGraph.dll"` (workaround c.790 appliqué) : plus aucun chemin absolu po-2027-spécifique dans le notebook. Le répertoire `_deps/` est gitignore ; sur une nouvelle machine, copiez-y la DLL depuis le cache NuGet — la cellule prelude ci-dessous imprime le chemin source attendu.

**Limitation parse-time** : la directive `#r "..."` est resolue au parse-time de la cellule (avant execution runtime), donc elle exige une chaine litterale. Ni variable ni interpolation — c'est pourquoi le workaround passe par un chemin relatif, resolu depuis le dossier du notebook (kernel lance avec `probes/` en cwd).

### Workaround relatif (mesure first-hand c.790, appliqué c.803)

Copier la DLL vers `./_deps/QuikGraph.dll` puis utiliser `#r "./_deps/QuikGraph.dll"` (path relatif). Mesure c.790 : `OK_relative: QuikGraph.AdjacencyGraph` charge sans erreur, identique au resultat du path absolu. Caveat : la copie locale doit etre rafraichie si la version de QuikGraph change.

### Sur une autre machine

1. Ayez QuikGraph 2.5.0 dans le cache NuGet (ex. `dotnet add package QuikGraph --version 2.5.0` dans un csproj satellite) — la cellule prelude affiche le chemin.
2. Copiez la DLL vers `scripts/notebook_tools/probes/_deps/QuikGraph.dll`.
3. Le probe E (`#r "nuget: CsvHelper"`) peut echouer OU reussir : le bug est **non déterministe** (voir probe E) — c'est precisement l'objet de la mesure.

Voir `docs/reference/dotnet-restore-rfc-17361.md` section Workaround pour le prechargement complet (proposition `dotnet_preload_packages.py`, non implémenté).


In [1]:
// PRELUDE - resolution du path QuikGraph.dll via le profil utilisateur
// Affiche la valeur a substituer dans la cellule probe D sur une autre machine.
var profile = Environment.GetFolderPath(Environment.SpecialFolder.UserProfile);
var expected = System.IO.Path.Combine(profile, ".nuget/packages/quikgraph/2.5.0/lib/netstandard2.0/QuikGraph.dll");
Console.WriteLine($"QuikGraph.dll attendu a : {expected}");
Console.WriteLine($"Existe ? {System.IO.File.Exists(expected)}");


The below script needs to be able to find the current output cell; this is an easy method to get it.

Existe ? True


In [2]:
// PROBE A — 1er #r nuget dans la session — ATTENDU: OK
#r "nuget: IKVM, 8.15.0"
Console.WriteLine("OK_A: IKVM 8.15.0 restore (1er restore nuget).");

Installed Packages IKVM, 8.15.0

OK_A: IKVM 8.15.0 restore (1er restore nuget).


In [3]:
// PROBE D - #r file.dll local - ATTENDU: OK
// By-pass du PackageRestoreContext pour les restores fichier.
// PORTABILITE : voir cellule prelude pour le path dynamique de votre machine.
//
// Workaround relatif c.790 (mesure first-hand OK) : copier la DLL vers
// `./_deps/QuikGraph.dll`, puis utiliser `#r "./_deps/QuikGraph.dll"`
// (chemin relatif, accepte par `#r` au parse-time).
//
// Workaround relatif APPLIQUE (c.803) : la DLL est copiee dans ./_deps/ (gitignore),
// le chemin relatif est resolu au parse-time depuis le dossier du notebook.
#r "./_deps/QuikGraph.dll"
Console.WriteLine($"OK_D: file.dll local restore works, type: {typeof(QuikGraph.AdjacencyGraph<int, QuikGraph.Edge<int>>).FullName}");


OK_D: file.dll local restore works, type: QuikGraph.AdjacencyGraph`2[[System.Int32, System.Private.CoreLib, Version=9.0.0.0, Culture=neutral, PublicKeyToken=7cec85d7bea7798e],[QuikGraph.Edge`1[[System.Int32, System.Private.CoreLib, Version=9.0.0.0, Culture=neutral, PublicKeyToken=7cec85d7bea7798e]], QuikGraph, Version=2.5.0.0, Culture=neutral, PublicKeyToken=46bd58b0789759cb]]


In [4]:
// PROBE E — nuget APRÈS file.dll — mesure discriminante (c.760, re-mesurée c.807)
// c.760 (2026-09-22) : KO — System.ArgumentException: Must provide errors when
//   succeeded is false. (PackageRestoreContext.RestoreAsync). Sorties conservées
//   aux têtes 261d8aa709 et 03260ceae2 de cette branche ; le run Papermill du
//   23/09 01:42Z portait exception: true pour la même cause.
// c.803 (23/09 15:10Z) : OK — restore silencieux, CsvHelper 33.0.1 déjà en cache NuGet local.
// c.807 (23/09 17:44Z) : OK à CACHE FROID — 33.0.1 purgé avant le run, re-téléchargé
//   pendant, restore réussi quand même : l'hypothèse « cache chaud » est réfutée.
// Verdict : bug NON DÉTERMINISTE — 3 occurrences c.760, 0 sur les re-tentatives
//   c.803/c.807 ; précondition exacte inconnue (race candidate, RFC §Cause racine).
// Ce WriteLine ne s'affiche QUE si le restore a réussi ; si la cellule meurt sur
// l'ArgumentException, c'est l'output error qui porte la mesure.
#r "nuget: CsvHelper, 33.0.1"
Console.WriteLine("E_MEASURED_OK: nuget restore SUCCEEDED this session (no ArgumentException) - c.803/c.807; see RFC Mesures.");


Installed Packages CsvHelper, 33.0.1

E_MEASURED_OK: nuget restore SUCCEEDED this session (no ArgumentException) - c.803/c.807; see RFC Mesures.


## Conclusion (mesures c.760, re-mesures c.803/c.807)

| Probe | Résultat | Diagnostic |
|---|---|---|
| A (1er `#r nuget`) | OK | `PackageRestoreContext` à l'état initial |
| D (`#r "file.dll"`) | OK | by-pass du PackageRestoreContext |
| E (`#r "nuget:"` après D) | KO c.760 / OK c.803, c.807 | `ArgumentException` **non déterministe** — voir probe E et RFC §Mesures |

**Lecture honnête** : le bug (2ᵉ restore NuGet d'une session qui lève `Must provide errors when succeeded is false`) est **réel** — il s'est produit 3 fois le 2026-09-22 (c.760, y compris le run Papermill archivé dans l'historique de cette branche) — mais il n'a **pas** été reproduit sur les 2 re-tentatives du 2026-09-23, dont une à cache NuGet froid. La précondition exacte est inconnue ; la race soupçonnée dans `RestoreAsync()` (RFC §Cause racine) reste l'explication la plus cohérente.

**Workaround défensif** (recommandé tant que le bug n'est pas corrigé upstream) : un notebook .NET qui doit charger plus d'un package reste plus sûr en référençant des assemblies locales (`#r "file.dll"` — voir la piste relative `_deps/` ci-dessus) : quand l'`ArgumentException` se produit, elle tue la cellule sans contournement runtime possible.

**Fix root cause** : bug interne `Microsoft.DotNet.Interactive.PackageManagement.PackageRestoreResult..ctor` — out-of-scope local.
